# recruitment agent (use case 1)
An AI-powered recruitment pipeline that automates resume analysis, candidate screening, scheduling, and offer letter generation using **CrewAI**.

### step 1: install dependencies
Install the required Python packages for document processing (PDF and DOCX) and PDF generation.

In [1]:
# Install necessary packages for file reading
%pip install docx2txt pypdf reportlab


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### step 2: initialization & imports
Set up the environment, import necessary libraries, and define utility functions to read resume and job description files.

In [2]:
import os
import csv
import smtplib
import ssl
from email.message import EmailMessage
import docx2txt
from pypdf import PdfReader
from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import letter
from crewai import Agent, Task, Crew, Process
from crewai.tools import tool
from langchain.tools import tool as langchain_tool

CSV_FILE = 'candidates.csv'

# --- Original Content ---
import os
import docx2txt
from pypdf import PdfReader

# Function to read PDF
def read_pdf(file_path):
    reader = PdfReader(file_path)
    text = ""
    for page in reader.pages:
        text += page.extract_text()
    return text

# Function to read DOCX
def read_docx(file_path):
    return docx2txt.process(file_path)

# Paths to data files
resume_path = 'data/Akhil_Peram__Resume___Copy_ (3).pdf'
jd_path = 'data/google_genai_fresher_jd.docx'

# Read Content
resume_content = ""
jd_content = ""

if os.path.exists(resume_path):
    try:
        resume_content = read_pdf(resume_path)
        print("Resume read successfully.")
    except Exception as e:
        print(f"Error reading resume: {e}")
else:
    print(f"Resume file not found at {resume_path}")

if os.path.exists(jd_path):
    try:
        jd_content = read_docx(jd_path)
        print("JD read successfully.")
    except Exception as e:
        print(f"Error reading JD: {e}")
else:
    print(f"JD file not found at {jd_path}")

Resume read successfully.
JD read successfully.


### step 3: define custom tools
Custom tools enable the agents to interact with the database (CSV), send emails, ask for manager approval, and generate offer letters.

In [3]:
import csv
import os
import smtplib
import ssl
from email.message import EmailMessage
from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import letter
from crewai.tools import tool

# --- TOOLS ---

# Existing Tools

@tool("AddCandidateDetails")
def add_candidate_details(name: str, email: str, phone: str, experience: str, skills: str):
    """Adds extracted candidate details to the candidates.csv file."""
    if not os.path.exists(CSV_FILE):
        with open(CSV_FILE, mode='w', newline='') as file:
            writer = csv.writer(file)
            writer.writerow(['Name', 'Email', 'Phone', 'Experience', 'Skills', 'Score', 'Status', 'Justification'])
            
    name = str(name).strip()
    email = str(email).strip()
    phone = str(phone).strip()
    
    with open(CSV_FILE, mode='a', newline='') as file:
        writer = csv.writer(file)
        writer.writerow([name, email, phone, experience, skills, '', '', ''])
    return "Candidate details extracted and saved."

@tool("UpdateCandidateStatus")
def update_candidate_status(name: str, score: int, status: str, justification: str):
    """Updates the score, status, and justification for an existing candidate in the candidates.csv file."""
    rows = []
    updated = False
    if not os.path.exists(CSV_FILE):
        return "CSV file not found."
    
    with open(CSV_FILE, mode='r', newline='') as file:
        reader = csv.reader(file)
        try:
            header = next(reader)
        except StopIteration:
            return "CSV file is empty."
        rows.append(header)
        for row in reader:
            if row and row[0].strip().lower() == name.strip().lower():
                if len(row) < 8:
                    row.extend([''] * (8 - len(row)))
                row[5] = str(score)
                row[6] = status
                row[7] = justification
                updated = True
            rows.append(row)
    
    if updated:
        with open(CSV_FILE, mode='w', newline='') as file:
            writer = csv.writer(file)
            writer.writerows(rows)
        return "Candidate status updated successfully."
    else:
        return f"Candidate {name} not found to update."

@tool("GetCandidateDetails")
def get_candidate_details(name: str):
    """Fetches candidate email and status from candidates.csv using the name."""
    if not os.path.exists(CSV_FILE):
        return "CSV file not found."
    
    with open(CSV_FILE, mode='r', newline='') as file:
        reader = csv.reader(file)
        next(reader, None) # Skip header
        for row in reader:
            if row and row[0].strip().lower() == name.strip().lower():
                return f"Name: {row[0]}, Email: {row[1]}, Status: {row[6]}"
    return f"Candidate {name} not found in CSV."

# New / Updated Tools

@tool("AskHiringManager")
def ask_hiring_manager(candidate_name: str, score: int, justification: str):
    """Asks the hiring manager if the candidate is Selected or Not Selected. If Selected, asks for CTC."""
    print(f"\n--- HIRING MANAGER REVIEW ---\nCandidate: {candidate_name}\nScore: {score}\nJustification: {justification}\n")
    
    decision = input("Is the candidate SELECTED? (yes/no): ").strip().lower()
    
    if decision == 'yes':
        ctc = input("Enter CTC offered (e.g. 50 LPA): ").strip()
        return f"Selected|{ctc}"
    else:
        return "Not Selected"

@tool("GenerateOfferLetter")
def generate_offer_letter(name: str, ctc: str):
    """Generates a PDF offer letter for the selected candidate."""
    file_name = f"Offer_Letter_{name.replace(' ', '_')}.pdf"
    try:
        c = canvas.Canvas(file_name, pagesize=letter)
        c.drawString(100, 750, "OFFER LETTER")
        c.drawString(100, 700, f"Dear {name},")
        c.drawString(100, 650, "We are pleased to offer you the position of GenAI Software Engineer at Google.")
        c.drawString(100, 600, f"Your annual CTC will be: {ctc}")
        c.drawString(100, 550, "We look forward to having you on board!")
        c.drawString(100, 500, "Sincerely,")
        c.drawString(100, 480, "Google Recruitment Team")
        c.save()
        return f"Offer letter generated: {file_name}"
    except Exception as e:
        return f"Failed to generate offer letter: {str(e)}"

@tool("SendEmailTool")
def send_email_tool(recipient_email: str, subject: str, body: str, attachment_path: str = None):
    """Sends an email. Supports attachments. Auto-detects SSL (465) vs STARTTLS (587)."""
    smtp_host = os.getenv('SMTP_HOST')
    smtp_port = os.getenv('SMTP_PORT')
    smtp_user = os.getenv('SMTP_USER')
    smtp_password = os.getenv('SMTP_PASSWORD')
    
    recipient_email = str(recipient_email)
    
    if not all([smtp_host, smtp_port, smtp_user, smtp_password]):
        print(f"\n[MOCK EMAIL SENT]\nTo: {recipient_email}\nSubject: {subject}\nBody:\n{body}\nAttachment: {attachment_path}\n")
        return "Mock email sent successfully (SMTP credentials missing)."
    
    try:
        msg = EmailMessage()
        msg.set_content(body)
        msg['Subject'] = subject
        msg['From'] = smtp_user
        msg['To'] = recipient_email
        
        if attachment_path and os.path.exists(attachment_path):
            with open(attachment_path, 'rb') as f:
                file_data = f.read()
                file_name = os.path.basename(attachment_path)
                msg.add_attachment(file_data, maintype='application', subtype='pdf', filename=file_name)
        
        port = int(smtp_port)
        # Use unverified context to avoid SSL errors on local env
        context = ssl._create_unverified_context()
        
        if port == 465:
            with smtplib.SMTP_SSL(smtp_host, port, context=context) as server:
                server.login(smtp_user, smtp_password)
                server.send_message(msg)
        else:
            with smtplib.SMTP(smtp_host, port) as server:
                server.starttls(context=context)
                server.login(smtp_user, smtp_password)
                server.send_message(msg)
                
        return f"Email sent successfully to {recipient_email}."
    except Exception as e:
        return f"Failed to send email: {str(e)}"


### step 4: define agents
Define specialized agents with unique roles, goals, and backstories to handle different stages of the recruitment process.

In [4]:
# --- AGENTS ---

# Define Resume Analyst Agent
resume_analyst = Agent(
    role='Resume Analyst',
    goal='Extract candidate details and match resumes against job descriptions.',
    backstory=(
        "You are an expert technical recruiter. "
        "First, you extract key details from a resume to maintain a database. "
        "Then, you objectively evaluate the candidate against the Job Description."
    ),
    verbose=True,
    allow_delegation=False,
    tools=[add_candidate_details, update_candidate_status]
)

# Define Interview Scheduler Agent
interview_scheduler = Agent(
    role='Interview Scheduler',
    goal='Communicate accurately with candidates based on their recruitment status.',
    backstory=(
        "You are a detailed-oriented Interview Scheduler. "
        "You ensure candidates receive timely and professional updates. "
        "You ALWAYS verify the candidate's email from the database before sending communications."
    ),
    verbose=True,
    allow_delegation=False,
    tools=[get_candidate_details, send_email_tool]
)

# Define Offer Manager Agent
offer_manager = Agent(
    role='Offer Manager',
    goal='Manage the final offer rollout process with human oversight.',
    backstory=(
        "You are the HR finter. You interact with the Hiring Manager to confirm the selection. "
        "If selected, you generate the offer letter and send it. "
        "if not, you ensure the candidate is informed."
    ),
    verbose=True,
    allow_delegation=False,
    tools=[ask_hiring_manager, generate_offer_letter, send_email_tool, get_candidate_details]
)

### step 5: define tasks
Assign specific tasks to each agent, detailing the scoring criteria and desired output for candidate evaluation.

In [5]:
# --- TASKS ---

# Task 1: Extraction
extraction_task = Task(
    description=(
        "Read the resume content below:\n{resume}\n\n"
        "1. Extract the following details: Name, Email, Phone Number, Experience (Years), and Key Skills.\n"
        "2. Use the `AddCandidateDetails` tool to save these details to the CSV file.\n"
        "Note: Ensure the Name is extracted accurately as it will be used for updating the status later."
    ),
    expected_output="Candidate details saved to CSV.",
    agent=resume_analyst
)

# Task 2: Matching
matching_task = Task(
    description=(
        "Compare the candidate's Resume against the Job Description (JD) to ensure a precise match.\n"
        "**Job Description**:\n{jd}\n\n"
        "**Resume**:\n{resume}\n\n"
        "**Scoring Criteria**:\n"
        "1. **Skills (40%)**: Match technical skills mentioned in the JD. Look for exact matches and synonyms.\n"
        "2. **Experience (30%)**: Verify relevant industry experience and years of experience.\n"
        "3. **Education (20%)**: Check if the candidate matches the educational background instructions.\n"
        "4. **Culture/Soft Skills (10%)**: Assess project leadership, collaboration, and other soft skills.\n\n"
        "**Instructions**:\n"
        "1. Perform a detailed analysis based on the criteria above.\n"
        "2. Calculate the total score out of 100.\n"
        "3. Determine the status: 'Shortlisted' ONLY if the Weighted Score is > 80. Otherwise 'Not Shortlisted'.\n"
        "4. Provide a strictly evidence-based justification. Mention specifically what matched and what is missing.\n"
        "5. Use the `UpdateCandidateStatus` tool to update the CSV record for the candidate (using their Name extracted earlier)."
    ),
    expected_output="Report containing Candidate Name, Score, Status, and Justification (for next agent context).",
    agent=resume_analyst
)

# Task 3: Email Communication
email_task = Task(
    description=(
        "Based on the output of the matching task:\n"
        "1. Identify the Candidate Name.\n"
        "2. Use `GetCandidateDetails` to retrieve the Email and confirmed Status from the CSV.\n"
        "3. Compose a professional email:\n"
        "   - If Status is 'Shortlisted': Subject: 'Interview Invitation - GenAI Engineer'. Body: Congratulate and invite.\n"
        "   - If Status is 'Not Shortlisted': Subject: 'Update on your Application'. Body: Thank you, but we are proceeding with other candidates.\n"
        "4. Use `SendEmailTool` to send the email.\n"
        "IMPORTANT: You MUST explicitly output 'STATUS: Shortlisted' or 'STATUS: Not Shortlisted' so the next step knows whether to proceed."
    ),
    expected_output="STATUS: [Status]. Email sent to [Email].",
    agent=interview_scheduler,
    context=[matching_task]
)

# Task 4: Offer Generation & Communication (Run only if Shortlisted)
offer_task = Task(
    description=(
        "**Offer Process**\n"
        "1. Identify Candidate Name. CRITICAL: Use `GetCandidateDetails` tool to get the correct Email from the CSV.\n"
        "2. Use `AskHiringManager` to present the Score/Justification and get the Final Decision.\n"
        "3. IF Decision is 'Selected':\n"
        "   - Get the 'CTC' from the manager tool output.\n"
        "   - Use `GenerateOfferLetter` to create the PDF.\n"
        "   - Use `SendEmailTool` to send an email with Subject: 'Offer Letter - Google' and attach the PDF.\n"
        "4. IF Decision is 'Not Selected':\n"
        "   - Use `SendEmailTool` to send a rejection email.\n"
    ),
    expected_output="Final decision executed (Offer sent or Rejection sent).",
    agent=offer_manager,
    context=[matching_task, email_task]
)


### step 6: assemble & kickoff crew
Organize the agents and tasks into crews. The process follows a sequential workflow, with the offer phase triggering only if a candidate is shortlisted.

In [6]:
# --- SPLIT CREW EXECUTION ---

# Crew 1: Screening & Email
screening_crew = Crew(
    agents=[resume_analyst, interview_scheduler],
    tasks=[extraction_task, matching_task, email_task],
    verbose=True,
    process=Process.sequential
)

# Crew 2: Offer (Conditional)
offer_crew = Crew(
    agents=[offer_manager],
    tasks=[offer_task],
    verbose=True,
    process=Process.sequential
)

inputs = {
    'resume': resume_content,
    'jd': jd_content
}

if resume_content and jd_content:
    print("--- Starting Screening Phase ---")
    result_1 = screening_crew.kickoff(inputs=inputs)
    print("\n--- Screening Phase Complete ---\n")
    print(result_1)
    
    # Check for 'Shortlisted' status in the output of the email task (result_1)
    # result_1 is usually a string output of the last task.
    # We can also check the actual candidates.csv if needed, but result check is faster.
    
    if "STATUS: Shortlisted" in str(result_1) or "Status: Shortlisted" in str(result_1):
        print("\n>>> Candidate SHORTLISTED. Proceeding to Offer Phase... <<<\n")
        result_2 = offer_crew.kickoff(inputs=inputs)
        print("\n--- Offer Phase Complete ---")
        print(result_2)
    else:
        print("\n>>> Candidate NOT SHORTLISTED. Process Stopped. Agent 3 (Offer) will NOT run. <<<\n")
else:
    print("Cannot run crew: Missing resume or JD content.")

--- Starting Screening Phase ---


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  16d7b305-efc9-4f97-acfd-b3224ca822b6                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Read the resume content below:                                                                           │
│  AKHIL KUMAR PERAM                                                                                              │
│  /linkedinlinkedin.com | /githubgithub.com                                                                      │
│  Gmail: akhilkumarperam@gmail.com                                                                               │
│  Ph: 9490639831                                                                                                 │
│  Education                                                                                                      │
│  Program Institution %/CGPA Year                                                                                │
│  M.Tech in Computational                                                                                        │
│  M.Tech in Aerospace Engineering                                                                                │
│  B.Tech. in Mechanical Engineering                                                                              │
│  XIIth std. - BIEAP                                                                                             │
│  Xth Std. - BSEAP                                                                                               │
│  Indian Institute of Technology Madras                                                                          │
│  Indian Institute of Technology Kanpur                                                                          │
│  Hindustan Institute of Technology                                                                              │
│  Sri Chaitanya JR College                                                                                       │
│  Ratnam EM High School                                                                                          │
│  8.47                                                                                                           │
│  7.25                                                                                                           │
│  7.84                                                                                                           │
│  94.6%                                                                                                          │
│  8.8                                                                                                            │
│  2023 - 2025                                                                                                    │
│  2021 - 2023                                                                                                    │
│  2016 - 2020                                                                                                    │
│  2014 - 2016                                                                                                    │
│  2014                                                                                                           │
│  Technical Skills                                                                                               │
│  Languages/ Tools : Python, SQL, MATLAB, Excel, GitHub                                                          │
│  Tools/Frameworks: Numpy, Pandas, Matplotlib, Seaborn, Scikit-Learn, Tensorflow, NLTK, Langchain, LangSmith,    │
│  Streamlit                                                                                                      │
│  Core Topics: Machine Learning, Deep Learning, NLP , LL

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Resume Analyst                                                                                          │
│                                                                                                                 │
│  Task: Read the resume content below:                                                                           │
│  AKHIL KUMAR PERAM                                                                                              │
│  /linkedinlinkedin.com | /githubgithub.com                                                                      │
│  Gmail: akhilkumarperam@gmail.com                                                                               │
│  Ph: 9490639831                                                                                                 │
│  Education                                                                                                      │
│  Program Institution %/CGPA Year                                                                                │
│  M.Tech in Computational                                                                                        │
│  M.Tech in Aerospace Engineering                                                                                │
│  B.Tech. in Mechanical Engineering                                                                              │
│  XIIth std. - BIEAP                                                                                             │
│  Xth Std. - BSEAP                                                                                               │
│  Indian Institute of Technology Madras                                                                          │
│  Indian Institute of Technology Kanpur                                                                          │
│  Hindustan Institute of Technology                                                                              │
│  Sri Chaitanya JR College                                                                                       │
│  Ratnam EM High School                                                                                          │
│  8.47                                                                                                           │
│  7.25                                                                                                           │
│  7.84                                                                                                           │
│  94.6%                                                                                                          │
│  8.8                                                                                                            │
│  2023 - 2025                                                                                                    │
│  2021 - 2023                                                                                                    │
│  2016 - 2020                                                                                                    │
│  2014 - 2016                                                                                                    │
│  2014                                                                                                           │
│  Technical Skills                                                                                               │
│  Languages/ Tools : Python, SQL, MATLAB, Excel, GitHub                                                          │
│  Tools/Frameworks: Numpy, Pandas, Matplotlib, Seaborn, Scikit-Learn, Tensorflow, NLTK, Langchain, LangSmith,    │
│  Streamlit                                             

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: add_candidate_details                                                                                    │
│  Args: {'name': 'AKHIL KUMAR PERAM', 'email': 'akhilkumarperam@gmail.com', 'phone': '9490639831',               │
│  'experience': 'Relevant experience from August 2024 to Present (approx 1 year) including roles like            │
│  Technical...                                                                                                   │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────── ✅ Tool Execution Completed (#2) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool Completed                                                                                                 │
│  Tool: send_email_tool                                                                                          │
│  Output: Email sent successfully to akhilkumarperam@gmail.com.                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Offer Manager                                                                                           │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  The candidate AKHIL KUMAR PERAM has been selected with a CTC of 15 LPA. The offer letter has been generated    │
│  and sent via email to akhilkumarperam@gmail.com with the subject "Offer Letter - Google." The final decision   │
│  to extend the offer has been executed successfully.                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  **Offer Process**                                                                                              │
│  1. Identify Candidate Name. CRITICAL: Use `GetCandidateDetails` tool to get the correct Email from the CSV.    │
│  2. Use `AskHiringManager` to present the Score/Justification and get the Final Decision.                       │
│  3. IF Decision is 'Selected':                                                                                  │
│     - Get the 'CTC' from the manager tool output.                                                               │
│     - Use `GenerateOfferLetter` to create the PDF.                                                              │
│     - Use `SendEmailTool` to send an email with Subject: 'Offer Letter - Google' and attach the PDF.            │
│  4. IF Decision is 'Not Selected':                                                                              │
│     - Use `SendEmailTool` to send a rejection email.                                                            │
│                                                                                                                 │
│  Agent:                                                                                                         │
│  Offer Manager                                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  3a3fedb7-f4dc-4828-99bd-e9e8d46046fb                                                                           │
│  Final Output: The candidate AKHIL KUMAR PERAM has been selected with a CTC of 15 LPA. The offer letter has     │
│  been generated and sent via email to akhilkumarperam@gmail.com with the subject "Offer Letter - Google." The   │
│  final decision to extend the offer has been executed successfully.                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


--- Offer Phase Complete ---
The candidate AKHIL KUMAR PERAM has been selected with a CTC of 15 LPA. The offer letter has been generated and sent via email to akhilkumarperam@gmail.com with the subject "Offer Letter - Google." The final decision to extend the offer has been executed successfully.
